# Representación molecular

Un modelo no puede recibir un SMILES directamente: hay que convertir la estructura en un
vector numérico. La elección de representación suele influir más en el resultado que la
elección de algoritmo.

## Opciones

| Representación | Qué codifica | Dimensión | Interpretable | Nota |
|---|---|---|---|---|
| **Morgan / ECFP** | subestructuras circulares por átomo | 1024–4096 bits | sí, vía bit→fragmento | el estándar de facto en QSAR |
| **RDKit FP** | caminos lineales | 2048 bits | parcialmente | complementa a ECFP |
| **MACCS keys** | 166 subestructuras predefinidas | 166 | muy | pequeño y rápido, menos expresivo |
| **Avalon** | caminos + features | 512–1024 | parcialmente | buen rendimiento empírico |
| **Descriptores RDKit** | ~210 propiedades calculadas | ~210 | sí | MW, logP, TPSA, índices topológicos |
| **Grafo molecular** | átomos y enlaces, aprendido | — | con atribución | para GNN; necesita más datos |

El plan: **ECFP4 (Morgan radio 2, 2048 bits) + descriptores fisicoquímicos** como
representación base, y grafo molecular para la variante con GNN.

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator, MACCSkeys
from rdkit.Chem import rdMolDescriptors
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
RANDOM_STATE = 42

df = pd.read_parquet("../datos/procesados/dataset_modelado.parquet")
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
assert all(m is not None for m in mols), "hay SMILES que no parsean"
len(mols)

## Fingerprints de Morgan (ECFP4)

Cada bit indica la presencia de un entorno circular de radio ≤ 2 alrededor de algún
átomo. Se guarda también el mapa bit → fragmento, necesario para interpretar el modelo
en el capítulo 8.

In [ ]:
FP_SIZE = 2048
RADIUS = 2

gen = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=FP_SIZE)

X_fp = np.zeros((len(mols), FP_SIZE), dtype=np.uint8)
bit_info = []

for i, m in enumerate(mols):
    ao = rdFingerprintGenerator.AdditionalOutput()
    ao.AllocateBitInfoMap()
    fp = gen.GetFingerprint(m, additionalOutput=ao)
    X_fp[i] = np.frombuffer(fp.ToBitString().encode(), "u1") - ord("0")
    bit_info.append(ao.GetBitInfoMap())

print("matriz de fingerprints:", X_fp.shape)
print("densidad media de bits activos:", X_fp.mean())

In [ ]:
# Bits nunca activados: no aportan nada y se pueden eliminar
activos = X_fp.sum(axis=0)
print(f"bits siempre a 0: {(activos == 0).sum()}")
print(f"bits en <5 moléculas: {(activos < 5).sum()}")

mask_bits = activos >= 5          # umbral de frecuencia mínima
X_fp_red = X_fp[:, mask_bits]
print("tras filtrar:", X_fp_red.shape)

## Descriptores fisicoquímicos

RDKit calcula ~210 descriptores. Muchos están muy correlacionados entre sí; se filtran
por varianza nula y correlación alta.

In [ ]:
nombres = [n for n, _ in Descriptors._descList]
calc = Chem.Descriptors.CalcMolDescriptors

desc = pd.DataFrame([calc(m) for m in mols], index=df.index)

# limpieza
desc = desc.replace([np.inf, -np.inf], np.nan)
desc = desc.dropna(axis=1, how="any")
desc = desc.loc[:, desc.std() > 0]

# quitar descriptores redundantes
c = desc.corr().abs()
sup = c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
redundantes = [col for col in sup.columns if (sup[col] > 0.95).any()]
desc = desc.drop(columns=redundantes)

print(f"descriptores conservados: {desc.shape[1]} (se eliminaron {len(redundantes)} redundantes)")
desc.head()

## Matriz final

Los descriptores necesitan escalado (tienen rangos muy distintos); los bits del
fingerprint no. El escalador se ajusta **solo con el conjunto de entrenamiento**, en el
capítulo siguiente, para no filtrar información del test.

In [ ]:
import scipy.sparse as sp
from pathlib import Path

salida = Path("../datos/procesados")
salida.mkdir(parents=True, exist_ok=True)

np.save(salida / "X_fp.npy", X_fp_red)
desc.to_parquet(salida / "X_desc.parquet")
df[["chembl_id", "smiles", "pic50", "scaffold"]].to_parquet(salida / "y_meta.parquet")

print("guardado:")
print(" X_fp.npy        ", X_fp_red.shape)
print(" X_desc.parquet  ", desc.shape)

## Similitud y vecindario

La distancia de Tanimoto sobre fingerprints es la medida de similitud química estándar.
Se usará en el capítulo 7 para definir el dominio de aplicabilidad del modelo.

In [ ]:
from rdkit import DataStructs

fps = [gen.GetFingerprint(m) for m in mols]

# similitud del primer compuesto frente a todos
sims = DataStructs.BulkTanimotoSimilarity(fps[0], fps)
sims = np.array(sims)

print(f"compuesto de referencia: {df['chembl_id'].iloc[0]}")
print(f"similitud mediana al resto: {np.median(sims[1:]):.3f}")
print(f"vecinos con Tanimoto > 0,7: {(sims[1:] > 0.7).sum()}")